In [0]:
# 1. Generate Normal User Traffic (100,000 rows)
print("Generating normal user traffic...")
normal_traffic_df = spark.range(0, 100000).selectExpr(
    "uuid() as event_id",
    "cast(rand() * 10000 as int) as user_id", # 10,000 unique normal users
    # Safe Math: Subtract up to 7 days (604,800 seconds) from current time
    "timestamp(unix_timestamp(current_timestamp()) - cast(rand() * 604800 as int)) as event_time", 
    "case when rand() < 0.7 then 'view' when rand() < 0.9 then 'cart' else 'purchase' end as event_type",
    "cast(rand() * 500 as int) as product_id",
    "concat(cast(rand() * 255 as int), '.', cast(rand() * 255 as int), '.1.1') as ip_address"
)

# 2. Inject Bot Traffic (1,000 rows in under 60 seconds)
print("Injecting malicious bot traffic...")
bot_traffic_df = spark.range(0, 1000).selectExpr(
    "uuid() as event_id",
    "99999 as user_id", # This is our targeted bot ID
    # Safe Math: Subtract up to 60 seconds from current time
    "timestamp(unix_timestamp(current_timestamp()) - cast(rand() * 60 as int)) as event_time", 
    "'view' as event_type",
    "cast(rand() * 50 as int) as product_id",
    "'192.168.1.100' as ip_address" # Static IP
)

# 3. Combine them together
final_raw_df = normal_traffic_df.union(bot_traffic_df)

# 4. Define your exact S3 raw path
raw_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/raw/"

# 5. Write out the synthetic dataset as a CSV file to your S3 bucket
print(f"Writing dataset to {raw_s3_path} ...")
final_raw_df.write.format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(raw_s3_path)

print("✅ Data successfully generated and saved to S3!")

# Show a sample of the data we just created
display(final_raw_df)

In [0]:
# Wipe the slate clean for the real Kaggle data
paths_to_clear = [
    "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/raw/", # NOTE: Make sure your new Kaggle CSV is uploaded AFTER you clear this, or just manually delete the old synthetic CSV!
    "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/bronze/",
    "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/silver/",
    "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/gold/",
    "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/checkpoints/",
    "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/schema_data/"
]
for path in paths_to_clear:
    dbutils.fs.rm(path, True)
print("Pipeline reset! Ready for real data ingestion.")